# Releases and operational accounting

Original prose/data: CC BY-SA 4.0; code: Apache-2.0. This notebook recomputes release gates and small logical-clock policies. It does not deploy anything. It also reads the separately captured actual HTTP rollback evidence.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
assert (root / "code/knowledge-assistant").exists(), "Start the kernel at the book repository root"
sys.path.insert(0, str(root / "code/knowledge-assistant"))
sys.path.insert(0, str(root / "code/part-ix"))
from service import server
from support import QUERY, submit, http

from service_ops import manifest, TokenBucket, sli
from support import release_gate
import json


## Independent expectations reject a valid but wrong-date answer
The expected 750 comes from v2 and the original 2026 question, not from the candidate output.

In [2]:
good = release_gate(manifest())
bad = release_gate(manifest("rc-stale", historical_bug=True))
assert (good["passed_cases"], bad["passed_cases"]) == (10, 4)
print({"good": good["passed_cases"], "bad": bad["passed_cases"]})
print(next(row for row in bad["rows"] if row["case"] == "current"))


{'good': 10, 'bad': 4}
{'case': 'current', 'locale': 'en', 'request': {'question': 'Beijing lodging', 'locale': 'en', 'on_date': '2026-09-14', 'topic': 'travel'}, 'expected': {'status': 'answered', 'amount_yuan': 750, 'source': 'travel-v2'}, 'actual': {'status': 'answered', 'amount_yuan': 600, 'source': 'travel-v1'}, 'passed': False}


## Count request outcomes before dividing
These 20 request outcomes and logical arrival times are authored. Eighteen qualify as good and two do not. A 95% target permits one bad event, so two bad events consume twice that budget. This is not a production availability estimate.

In [3]:
rows = [dict(arrival=i, outcome="answered" if i < 18 else "error", eligible=True, duration=0.2) for i in range(20)]
report = sli(rows, 0, 60)
assert report["good"] == 18 and report["total"] == 20
assert abs(report["burn"] - 2) < 1e-12
print(report)
now = [0.0]
bucket = TokenBucket(2, 1, lambda: now[0])
allowed = []
for arrival in [0, 0, 0, 0.5, 1, 1.5, 2]:
    now[0] = arrival
    allowed.append(bucket.take("north")[0])
assert allowed == [True, True, False, False, True, False, True]
print(allowed)


{'window': [0, 60], 'total': 20, 'good': 18, 'bad': 2, 'sli': 0.9, 'target': 0.95, 'budget': 1.0000000000000009, 'burn': 1.9999999999999984, 'cancelled': 0, 'excluded': 0, 'alert': False}
[True, True, False, False, True, False, True]


## Inspect actual saved rollback, without repeating a side effect
The following records were captured by the loopback experiment runner. Loading this file is record replay, not a fresh HTTP measurement.

In [4]:
record = json.loads((root / "data/part-ix/release-run.json").read_text())
print({"failed": record["failed"]["result"]["answer"]["amount_yuan"],
       "restored": record["rollback"]["result"]["answer"]["amount_yuan"],
       "inflight": record["inflight"]["result"]["answer"]["amount_yuan"]})
assert record["failed"]["result"]["answer"]["amount_yuan"] == 600
assert record["rollback"]["result"]["answer"]["amount_yuan"] == 750
assert record["inflight"]["release_name"] == "rc-stale"


{'failed': 600, 'restored': 750, 'inflight': 600}


## Transfer exercise and visible answer
Can 18 good responses out of 18 retained successful traces establish 100% availability? No. Failed, rejected or unsampled requests may be missing. Define the eligible population and collection point, then count every outcome in that denominator. A sampled trace folder cannot supply a complete denominator by itself.